# Load Libraries and Data

In [ ]:
import pandas as pd
import numpy as np

radstar = pd.read_excel("radStar.xlsx")

In [ ]:
#Confirm sucessful load
radstar.head()

In [ ]:
#View info about the dataset
radstar.info()

In [ ]:
#View shape of dataset and save it for future comparison
original_size = radstar.shape
original_size

# Look for repeat data

The data had some repeat data points and times where the timestamps were intertwined between groups. This loop, while slow, finds all of the repeat packet IDs. 

In [ ]:
for idx, row in radstar.iterrows():
    #If the timestamp is happens before the timestamp one row up, print out the packetID
    if idx > 0 and (row["timestamp"] < radstar.iloc[idx-1]["timestamp"]):
        print(row["packetID"])

# Drop repeat data, nulls, and unecessary columns

In [ ]:
#Drop repeats (keeping interwoven ones for the time being)
#NOTE: Will fix group numbers later
#NOTE 2: These packetIDs were found from the above code, but they were organized manually for the below code

#Drop fully repeated packets
radstar = radstar[~radstar['packetID'].isin([533060, 553935, 555615, 
                                             555789, 556782, 563639, 
                                             567468, 567633, 574719, 
                                             575473, 576016, 578352])]

#Drop intertwined packets too
radstar = radstar[~radstar["packetID"].isin([538751, 555658, 556374, 
                                             564107, 564473, 567774, 
                                             572607, 575226, 575232, 
                                             579030, 580539])]

#Drop remaining nulls
#This works because all radiation detectors were null at the exact same time, so dropping all rows with null in `electron 0` gets all of them
radstar = radstar.dropna(subset="electron 0")

#View the shape after all the drops
radstar.shape

Drop `ses 0` because all values are either 0 or null

In [ ]:
radstar = radstar.drop("ses 0", axis=1)

# Fix Group Numbers
After dropping all those packets, the group numbers are now incorrect. This needs fixed

In [ ]:
radstar.reset_index(drop=True, inplace=True)

'''
#This is my original, unoptimized version of the code

cur_group = 1

for idx, row in radstar.iterrows():
    #If the time between timestamps is greater than or equal to 5 minutes, go to next group
    if idx > 0 and (row["timestamp"] - radstar.iloc[idx-1]["timestamp"]).total_seconds() >= 300:
        cur_group += 1
    radstar.at[idx, "Group"] = cur_group
'''

#Vectorized version
radstar["Group"] = radstar["timestamp"].diff().dt.total_seconds().ge(300).cumsum() + 1

# Create Calculated Fields

### Add columns for particles per second

In [ ]:
#Add an observation time column to calculate particles per second
#This column will be removed once we are done creating the new columns
radstar["observation_time"] = radstar["speedmode"].case_when(
    caselist=[
        (radstar["speedmode"] == 0, 4),
        (radstar["speedmode"] == 1, 16),
        (radstar["speedmode"] == 2, 64)
    ]
)

In [ ]:
radstar["proton0_ps"] = radstar["proton 0"] / radstar["observation_time"]
radstar["electron0_ps"] = radstar["electron 0"] / radstar["observation_time"]
# radstar["electron_per_sec_1"] = radstar["electron 1"] / radstar["observation_time"] #I don't think electron1_ps is necessary
radstar["xray0_ps"] = radstar["xray 0"] / radstar["observation_time"]
radstar["xray1_ps"] = radstar["xray 1"] / radstar["observation_time"]
radstar["xray2_ps"] = radstar["xray 2"] / radstar["observation_time"]
radstar["xray3_ps"] = radstar["xray 3"] / radstar["observation_time"]
radstar["ses_ps"] = radstar["ses 1"] / radstar["observation_time"]

### Create New Total Radiation Column

In [ ]:
radstar["total_radiation"] = radstar["proton 0"] + radstar["electron 0"] + radstar["xray 0"]
radstar["total_radiation_ps"] = radstar["total_radiation"] / radstar["observation_time"]

### Create New Identifier Column

#### Notes to self:
Group 1 Sample 1 should be "001001"
Group 208 Sample 37 should be "208037"

First half is just string version of Group forced to be thre chars
Second half is from iteration throughout each group  
- This comes from separating by group and looping through the entire group

In [ ]:
# Old version
# cur_index = 0

# # f"{radstar.iloc[0]["Group"]:03d}{1:03d}"
# for idx, row in radstar.iterrows():
#     if idx > 0 and row["Group"] != radstar.at[idx - 1, "Group"]:
#         cur_index = 1
#     else:
#         cur_index += 1

#     radstar.at[idx, "temp"] = f"{row["Group"]:03d}{cur_index:03d}"


# Vectorized verison
radstar["temp"] = (
    radstar["Group"].astype(str).str.zfill(3) +
    (radstar.groupby("Group").cumcount() + 1).astype(str).str.zfill(3)
)

In [ ]:
#Remove observation time once we're done with it
radstar.drop("observation_time", axis=1, inplace=True)

# Clean up column names

In [ ]:
radstar.columns

In [ ]:
radstar.rename(columns={"proton 0": "proton0", "proton 1": "proton1", 
                "electron 0": "electron0", "electron 1": "electron1",
                "xray 0": "xray0", "xray 1": "xray1", "xray 2": "xray2",
                "xray 3": "xray3", "ses 1": "ses", "Group": "group",
                "Lat": "lat", "Lon": "lon", "Alt": "alt", 
                "In Shadow": "in_shadow"}, inplace=True)
                
#ses 1 is renamed to `ses` since ses 0 has been dropped

In [ ]:
radstar.columns

In [ ]:
radstar.shape

# Export cleaned data to a CSV

In [ ]:
#RENAME FILE ONCE CLEANING IS DONE

#USING NATHAN'S VERSION OF THE FILE
#radstar.to_csv("partial_clean_radstar.csv")

# Add Kp Indexes to the data

### Load clean file and kp file

In [ ]:
clean_rad = pd.read_excel("cleaned_data.xlsx")
kp = pd.read_excel("radstar_kp.xlsx")

### Clean the kp file so it's in a long data format

In [ ]:
#Create a new, empty DataFrame with the columns we plan to have in our cleaned DataFrame
new_kp = pd.DataFrame(columns=["timestamp", "kp", "Ap"])

#Loop through all rows of the original kp DataFrame
for idx, row in kp.iterrows():
    #Loop through each row 8 times to grab each of the 8 kp values individually
    for i in range(8):
        #Create a temporary, 1-row DataFrame that holds a timestamp, kp value, and Ap value so that our new dataset has only one kp value per row
        temp = pd.DataFrame([{
                "timestamp": pd.to_datetime({"year": [kp.iloc[idx, 0]],     #Column 0 is the year
                                             "month": [kp.iloc[idx, 1]],    #Column 1 is the month
                                             "day": [kp.iloc[idx, 2]],      #Column 2 is the day
                                             "hour": [3+(i*3)]              #This gives us every 3 hours starting at 3 (kp is measured 3-hourly)
                                            })[0], 
                "kp": kp.iloc[idx, 3+i],    #Grab the next kp value
                "Ap": kp.iloc[idx, -1]      #The last row is the Ap value. All 8 kp values will be stored with this value
        }])
        #Concatinate our temporary DataFrame with our new one
        new_kp = pd.concat([new_kp, temp], ignore_index=True)

# Export the new version of the kp data set to excel

In [ ]:
new_kp.to_excel("new_kp.xlsx", index=False)

# Add Kp Indexes to the data

#### HOW THE BELOW CODE WORKS:
- Both files are ordered chronologically (after cleaning clean_rad)
- Loop through the items in clean_rad until it encounters a kp value with a timestamp before the clean_rad row
    - If the clean_rad row happens before the kp timestamp, that means it's within that kp value's 3-hour block and will recieve that kp value
    - If the clean_rad row happens after the kp timestamp, we have made it past that kp index chronologically and can move onto the next one
- Once we made it to the end of the one of the DataFrames, we have assigned a kp value to all clean_rad rows

In [ ]:
kp_idx = 15         #The index for what kp value we are on starting with the beginning of our data (everything before kp_dix==15 is for lagging kp)
clean_idx = 0       #The index for which clean_rad row we are on

#Loop through the entire document until we're either out of kp values to use or out of rows to assign kp vlaues to
while (clean_idx < len(clean_rad)) and (kp_idx < len(new_kp)):
    if clean_rad.iloc[clean_idx, 4] < new_kp.iloc[kp_idx, 0]:                       #clean_rad column 4 is timestamp, while new_kp timestamp column 0 is timestamp
        clean_rad.loc[clean_rad.index[clean_idx], "kp"] = new_kp.iloc[kp_idx, 1]    #Assign the current kp value to the current clean_rad row
        
        # Add new columns with lagged kp values
        for lag in [1, 2, 4, 8, 16, 24]:
            if kp_idx > lag:
                clean_rad.loc[clean_rad.index[clean_idx], f"kp_lag{lag*3}"] = new_kp.iloc[kp_idx - lag, 1]
            else:
                clean_rad.loc[clean_rad.index[clean_idx], f"kp_lag{lag*3}"] = np.nan  
        
        clean_rad.loc[clean_rad.index[clean_idx], "Ap"] = new_kp.iloc[kp_idx, 2]    #Assign the current Ap value to the current clean_rad row
        clean_idx += 1                                                              #Move on to the next clean_rad row
    else:
        kp_idx += 1                                                                 #Move on to the next kp value

In [ ]:
clean_rad.head(120)

# Export new data with all cleaned data plus kp and Ap values

In [ ]:
clean_rad.to_excel("clean_data_kp.xlsx", index=False)

# Clean and Concatonate Temperature Data

## Preliminary look at temperature

### Load the data

In [ ]:
temps = pd.read_excel("Book1.xlsx", skiprows=1)

### View the data

In [ ]:
temps.head()

In [ ]:
temps.shape

In [ ]:
temps.radioViewID.value_counts()

In [ ]:
temps.isna().groupby(temps["radioViewID"])["S4 Temp"].sum()

In [ ]:
temps.groupby(temps["radioViewID"])["S4 Temp"].count()

In [ ]:
print("Percentage of nulls: ")
temps.isna().groupby(temps["radioViewID"])["S4 Temp"].sum()/temps.isna().groupby(temps["radioViewID"])["S4 Temp"].size()

In [ ]:
temps.dropna().head()

In [ ]:
temps.dropna().shape

In [ ]:
temps.dtypes

### These are the non-null temperature values that occured during our data window

In [ ]:
rad_temps = temps[(temps["gatewayTS"] > clean_rad.iloc[1].timestamp) & (temps["gatewayTS"] < clean_rad.iloc[-1].timestamp)].dropna()

In [ ]:
rad_temps

In [ ]:
rad_temps["radioViewID"].value_counts()

## Add temperature data to `clean_rad`

In [ ]:
temp_idx = 0        #The index for which temperature value we are on 
clean_idx = 0       #The index for which clean_rad row we are on

#Loop through the entire document until we're either out of temperature values to use or out of rows to assign temperature vlaues to
while (clean_idx < len(clean_rad)) and (temp_idx < len(rad_temps)):
    if clean_rad.iloc[clean_idx, 4] < rad_temps.iloc[temp_idx, 0]:                                  #clean_rad column 4 is timestamp, while rad_temps timestamp column 0 is timestamp
        clean_rad.loc[clean_rad.index[clean_idx], "recent_temp"] = rad_temps.iloc[temp_idx, 1]      #Assign the current temperature value to the current clean_rad row
        clean_rad.loc[clean_rad.index[clean_idx], "recent_radio"] = rad_temps.iloc[temp_idx, 2]     #Assign the current radioID value to the current clean_rad row
        clean_idx += 1                                                                              #Move on to the next clean_rad row
    else:
        temp_idx += 1 

In [ ]:
clean_rad.head()

In [ ]:
clean_rad.recent_radio.value_counts()

## Export temp data

In [ ]:
clean_rad.to_excel("radstar_temps.xlsx", index=False)